In [1]:
USE {
	dependencies("org.duckdb:duckdb_jdbc:1.5.5.1")
}

In [2]:
%use dataframe
// %use lets-plot
// %use kandy
// %use plotly

import kotlinx.serialization.json.Json

In [3]:
import org.jetbrains.kotlinx.dataframe.io.DbConnectionConfig
import org.jetbrains.kotlinx.dataframe.schema.DataFrameSchema

In [4]:
val db = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_time_control_results_2026-08-30/time-control.duckdb"
val url = "jdbc:duckdb:$db"
val username = "duckdb"
val password = "password"

val dbConfig = DbConnectionConfig(url, username, password)

val tableName = "game_results"

In [5]:
val extendedDb = "/Users/darronporter/Downloads/Dissertation/code/output/results/gcloud_extended_time_experiments_results_2026-08-31/extended_time.db"
val extendedUrl = "jdbc:duckdb:$extendedDb"

val extendedDbConfig = DbConnectionConfig(extendedUrl, username, password)

In [15]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
    WITH games AS (
          SELECT
                winner.winner.model      AS winner_model,
                winner.winner.timeDuration      AS winner_time_duration,
                winner.winner.name       AS winner_colour,

                winner.loser.model       AS loser_model,
                winner.loser.timeDuration       AS loser_time_duration,
                winner.loser.name        AS loser_colour,

                winner.winCondition
          FROM game_results
    ),

    matches AS (
        SELECT
            winner_model || ' | td=' ||
            winner_time_duration || ' | ' ||
            winner_colour AS winner_player,

            loser_model || ' | td=' ||
            loser_time_duration || ' | ' ||
            loser_colour AS loser_player,

            winCondition
        FROM games
    )

    PIVOT matches
    ON loser_player
    USING count(*)
    GROUP BY winner_player
    ORDER BY winner_player
  """.trimIndent()
)

kotlin-logging: initializing... active logger factory: Slf4jLoggerFactory


winner_player,MCTS | td=PT0.100S | BLACK,MCTS | td=PT0.100S | WHITE,MCTS | td=PT0.300S | BLACK,MCTS | td=PT0.300S | WHITE,MCTS | td=PT0.500S | BLACK,MCTS | td=PT0.500S | WHITE,MCTS | td=PT1S | BLACK,MCTS | td=PT1S | WHITE,MINIMAX | td=PT0.100S | BLACK,MINIMAX | td=PT0.100S | WHITE,MINIMAX | td=PT0.300S | BLACK,MINIMAX | td=PT0.300S | WHITE,MINIMAX | td=PT0.500S | BLACK,MINIMAX | td=PT0.500S | WHITE,MINIMAX | td=PT1S | BLACK,MINIMAX | td=PT1S | WHITE
MCTS | td=PT0.100S | BLACK,0,0,0,0,0,0,0,0,0,488,0,0,0,0,0,0
MCTS | td=PT0.100S | WHITE,0,0,0,0,0,0,0,0,488,0,0,0,0,0,0,0
MCTS | td=PT0.300S | BLACK,0,0,0,0,0,0,0,0,0,0,0,487,0,0,0,0
MCTS | td=PT0.300S | WHITE,0,0,0,0,0,0,0,0,0,0,496,0,0,0,0,0
MCTS | td=PT0.500S | BLACK,0,0,0,0,0,0,0,0,0,0,0,0,0,496,0,0
MCTS | td=PT0.500S | WHITE,0,0,0,0,0,0,0,0,0,0,0,0,496,0,0,0
MCTS | td=PT1S | BLACK,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,496
MCTS | td=PT1S | WHITE,0,0,0,0,0,0,0,0,0,0,0,0,0,0,496,0
MINIMAX | td=PT0.100S | BLACK,0,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0
MINIMAX | td=PT0.100S | WHITE,15,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [27]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",

    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') AS "MM (W) vs MCTS (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS')    AS "MM (W) vs MCTS (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS')    AS "MM (W) vs MCTS (B) MCTS Games",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MM (W) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MCTS (B) Win %",


    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS')    AS "MCTS (W) vs MM (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') AS "MCTS (W) vs MM (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX')    AS "MCTS (W) vs MM (B) Games",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MCTS (W) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MM (B) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,MM (W) vs MCTS (B) MM Win,MM (W) vs MCTS (B) MCTS Win,MM (W) vs MCTS (B) MCTS Games,MM (W) Win %,MCTS (B) Win %,MCTS (W) vs MM (B) MCTS Win,MCTS (W) vs MM (B) MM Win,MCTS (W) vs MM (B) Games,MCTS (W) Win %,MM (B) Win %,Combined MCTS Win %,Combined MM Win %,Total_Games
PT0.100S,PT0.100S,15,485,500,3.0%,97.0%,488,12,500,97.6%,2.4%,97.3%,2.7%,1000
PT0.300S,PT0.300S,13,487,500,2.6%,97.4%,491,9,500,98.2%,1.8%,97.8%,2.2%,1000
PT0.500S,PT0.500S,5,495,500,1.0%,99.0%,496,4,500,99.2%,0.8%,99.1%,0.9%,1000
PT1S,PT1S,4,496,500,0.8%,99.2%,495,5,500,99.0%,1.0%,99.1%,0.9%,1000
null,null,0,0,0,null,null,0,0,0,null,null,0.0%,0.0%,1


In [38]:
DataFrame.readSqlQuery(
	extendedDbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",

    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') AS "MM (W) vs MCTS (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS')    AS "MM (W) vs MCTS (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MCTS')    AS "MM (W) vs MCTS (B) MCTS Games",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MM (W) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MCTS' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MCTS'), 0), 4
    ) || '%' AS "MCTS (B) Win %",


    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS')    AS "MCTS (W) vs MM (B) MCTS Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') AS "MCTS (W) vs MM (B) MM Win",
    COUNT(*) FILTER (WHERE black_model = 'MINIMAX')    AS "MCTS (W) vs MM (B) Games",
    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MCTS') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MCTS (W) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE black_model = 'MINIMAX' AND winner_model = 'MINIMAX') * 100.0 /
        NULLIF(COUNT(*) FILTER (WHERE black_model = 'MINIMAX'), 0), 4
    ) || '%' AS "MM (B) Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "Combined MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,BLACK,MM (W) vs MCTS (B) MM Win,MM (W) vs MCTS (B) MCTS Win,MM (W) vs MCTS (B) MCTS Games,MM (W) Win %,MCTS (B) Win %,MCTS (W) vs MM (B) MCTS Win,MCTS (W) vs MM (B) MM Win,MCTS (W) vs MM (B) Games,MCTS (W) Win %,MM (B) Win %,Combined MCTS Win %,Combined MM Win %,Total_Games
PT1.500S,PT1.500S,MCTS,6,494,500,1.2%,98.8%,0,0,0,null,null,98.8%,1.2%,500
PT1.500S,PT1.500S,MINIMAX,0,0,0,null,null,497,3,500,99.4%,0.6%,99.4%,0.6%,500
PT1S,PT1S,MCTS,2,498,500,0.4%,99.6%,0,0,0,null,null,99.6%,0.4%,500
PT1S,PT1S,MINIMAX,0,0,0,null,null,496,4,500,99.2%,0.8%,99.2%,0.8%,500
PT2S,PT2S,MCTS,1,499,500,0.2%,99.8%,0,0,0,null,null,99.8%,0.2%,500
PT2S,PT2S,MINIMAX,0,0,0,null,null,500,0,500,100.0%,0.0%,100.0%,0.0%,500
PT3S,PT3S,MCTS,1,499,500,0.2%,99.8%,0,0,0,null,null,99.8%,0.2%,500
PT3S,PT3S,MINIMAX,0,0,0,null,null,498,2,500,99.6%,0.4%,99.6%,0.4%,500
null,null,null,0,0,0,null,null,0,0,0,null,null,0.0%,0.0%,7


In [44]:
DataFrame.readSqlQuery(
	extendedDbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    CASE WHEN black_model = 'MCTS' THEN 'MINIMAX' ELSE 'MCTS' END AS 'WHITE',
    black_model AS "BLACK",

    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    COUNT(*) AS "Games",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration, black_model, WHITE
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,WHITE,BLACK,MM Win,MCTS Win,Games,MCTS Win %,MM Win %,Total_Games
PT1.500S,PT1.500S,MCTS,MINIMAX,3,497,500,99.4%,0.6%,500
PT1.500S,PT1.500S,MINIMAX,MCTS,6,494,500,98.8%,1.2%,500
PT1S,PT1S,MINIMAX,MCTS,2,498,500,99.6%,0.4%,500
PT1S,PT1S,MCTS,MINIMAX,4,496,500,99.2%,0.8%,500
PT2S,PT2S,MINIMAX,MCTS,1,499,500,99.8%,0.2%,500
PT2S,PT2S,MCTS,MINIMAX,0,500,500,100.0%,0.0%,500
PT3S,PT3S,MINIMAX,MCTS,1,499,500,99.8%,0.2%,500
PT3S,PT3S,MCTS,MINIMAX,2,498,500,99.6%,0.4%,500
null,null,MCTS,null,0,0,7,0.0%,0.0%,7


In [43]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    CASE WHEN black_model = 'MCTS' THEN 'MINIMAX' ELSE 'MCTS' END AS 'WHITE',
    black_model AS "BLACK",

    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    COUNT(*) AS "Games",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration, black_model, WHITE
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,WHITE,BLACK,MM Win,MCTS Win,Games,MCTS Win %,MM Win %,Total_Games
PT0.100S,PT0.100S,MCTS,MINIMAX,12,488,500,97.6%,2.4%,500
PT0.100S,PT0.100S,MINIMAX,MCTS,15,485,500,97.0%,3.0%,500
PT0.300S,PT0.300S,MCTS,MINIMAX,9,491,500,98.2%,1.8%,500
PT0.300S,PT0.300S,MINIMAX,MCTS,13,487,500,97.4%,2.6%,500
PT0.500S,PT0.500S,MINIMAX,MCTS,5,495,500,99.0%,1.0%,500
PT0.500S,PT0.500S,MCTS,MINIMAX,4,496,500,99.2%,0.8%,500
PT1S,PT1S,MCTS,MINIMAX,5,495,500,99.0%,1.0%,500
PT1S,PT1S,MINIMAX,MCTS,4,496,500,99.2%,0.8%,500
null,null,MCTS,null,0,0,1,0.0%,0.0%,1


In [7]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,MM Win,MCTS Win,MCTS Win %,MM Win %,Total_Games
PT0.100S,PT0.100S,27,973,97.300000,2.700000,1000
PT0.300S,PT0.300S,22,978,97.800000,2.200000,1000
PT0.500S,PT0.500S,9,991,99.100000,0.900000,1000
PT1S,PT1S,9,991,99.100000,0.900000,1000
null,null,0,0,0.000000,0.000000,1


In [13]:
DataFrame.readSqlQuery(
	extendedDbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,MM Win,MCTS Win,MCTS Win %,MM Win %,Total_Games
PT1.500S,PT1.500S,9,991,99.100000,0.900000,1000
PT1S,PT1S,6,994,99.400000,0.600000,1000
PT2S,PT2S,1,999,99.900000,0.100000,1000
PT3S,PT3S,3,997,99.700000,0.300000,1000
null,null,0,0,0.000000,0.000000,7


In [65]:
DataFrame.readSqlQuery(
	dbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour,
        winner.winCondition AS winCondition
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour,
        winCondition
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    winCondition,
    CASE WHEN black_model = 'MCTS' THEN 'MINIMAX' ELSE 'MCTS' END AS 'WHITE',
    black_model AS "BLACK",

    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    COUNT(*) AS "Games",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration, black_model, WHITE, winCondition
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,winCondition,WHITE,BLACK,MM Win,MCTS Win,Games,MCTS Win %,MM Win %,Total_Games
PT0.100S,PT0.100S,CapturedAllGIPFPieces,MINIMAX,MCTS,10,33,43,76.7442%,23.2558%,43
PT0.100S,PT0.100S,MadeTheLastMove,MCTS,MINIMAX,5,441,446,98.8789%,1.1211%,446
PT0.100S,PT0.100S,CapturedAllGIPFPieces,MCTS,MINIMAX,7,47,54,87.037%,12.963%,54
PT0.100S,PT0.100S,MadeTheLastMove,MINIMAX,MCTS,5,452,457,98.9059%,1.0941%,457
PT0.300S,PT0.300S,MadeTheLastMove,MINIMAX,MCTS,3,433,436,99.3119%,0.6881%,436
PT0.300S,PT0.300S,MadeTheLastMove,MCTS,MINIMAX,2,412,414,99.5169%,0.4831%,414
PT0.300S,PT0.300S,CapturedAllGIPFPieces,MCTS,MINIMAX,7,79,86,91.8605%,8.1395%,86
PT0.300S,PT0.300S,CapturedAllGIPFPieces,MINIMAX,MCTS,10,54,64,84.375%,15.625%,64
PT0.500S,PT0.500S,MadeTheLastMove,MINIMAX,MCTS,2,427,429,99.5338%,0.4662%,429
PT0.500S,PT0.500S,CapturedAllGIPFPieces,MINIMAX,MCTS,3,68,71,95.7746%,4.2254%,71


In [66]:
DataFrame.readSqlQuery(
	extendedDbConfig,
	sqlQuery = """
WITH games AS (
    SELECT
        winner.winner.model      AS winner_model,
        winner.winner.timeDuration      AS winner_timeDuration,
        winner.winner.name       AS winner_colour,

        winner.loser.model       AS loser_model,
        winner.loser.timeDuration  AS loser_timeDuration,
        winner.loser.name        AS loser_colour,
        winner.winCondition AS winCondition
    FROM game_results
),

normalized AS (
    SELECT
        CASE WHEN winner_model = 'MCTS' THEN winner_timeDuration ELSE loser_timeDuration END AS mcts_timeDuration,
        CASE WHEN winner_model = 'MINIMAX' THEN winner_timeDuration ELSE loser_timeDuration END AS mm_timeDuration,

        CASE WHEN winner_colour = 'BLACK' THEN winner_model ELSE loser_model END AS black_model,
        winner_model,
        winner_colour,
        winCondition
    FROM games
)

SELECT
    mcts_timeDuration AS "MCTS TD",
    mm_timeDuration  AS "MM TD",
    winCondition,
    CASE WHEN black_model = 'MCTS' THEN 'MINIMAX' ELSE 'MCTS' END AS 'WHITE',
    black_model AS "BLACK",

    COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') AS "MM Win",
    COUNT(*) FILTER (WHERE winner_model = 'MCTS')    AS "MCTS Win",
    COUNT(*) AS "Games",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MCTS') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MCTS Win %",

    ROUND(
        COUNT(*) FILTER (WHERE winner_model = 'MINIMAX') * 100.0 /
        COUNT(*), 4
    ) || '%' AS "MM Win %",

    COUNT(*) as "Total_Games"

FROM normalized
GROUP BY mcts_timeDuration, mm_timeDuration, black_model, WHITE, winCondition
ORDER BY mcts_timeDuration ASC, mm_timeDuration ASC
    """.trimIndent()
)

MCTS TD,MM TD,winCondition,WHITE,BLACK,MM Win,MCTS Win,Games,MCTS Win %,MM Win %,Total_Games
PT1.500S,PT1.500S,CapturedAllGIPFPieces,MINIMAX,MCTS,4,98,102,96.0784%,3.9216%,102
PT1.500S,PT1.500S,MadeTheLastMove,MINIMAX,MCTS,2,396,398,99.4975%,0.5025%,398
PT1.500S,PT1.500S,CapturedAllGIPFPieces,MCTS,MINIMAX,3,110,113,97.3451%,2.6549%,113
PT1.500S,PT1.500S,MadeTheLastMove,MCTS,MINIMAX,0,387,387,100.0%,0.0%,387
PT1S,PT1S,MadeTheLastMove,MCTS,MINIMAX,1,412,413,99.7579%,0.2421%,413
PT1S,PT1S,MadeTheLastMove,MINIMAX,MCTS,1,409,410,99.7561%,0.2439%,410
PT1S,PT1S,CapturedAllGIPFPieces,MCTS,MINIMAX,3,84,87,96.5517%,3.4483%,87
PT1S,PT1S,CapturedAllGIPFPieces,MINIMAX,MCTS,1,89,90,98.8889%,1.1111%,90
PT2S,PT2S,CapturedAllGIPFPieces,MCTS,MINIMAX,0,102,102,100.0%,0.0%,102
PT2S,PT2S,CapturedAllGIPFPieces,MINIMAX,MCTS,0,96,96,100.0%,0.0%,96
